# Feature Engineering

Leakage-safe feature engineering. Formula features are applied identically to train/validation/test. Category levels for one-hot encoding are learned from `train_split_preprocessed` only, then applied to validation and test. `health_condition` is used only for train-side feature review and remains the target.

In [1]:
import re

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

## Load Preprocessed Splits

In [2]:
ID_COL = "id"
TARGET_COL = "health_condition"

train_base = pd.read_csv("data/train_split_preprocessed.csv")
val_base = pd.read_csv("data/val_split_preprocessed.csv")
test_base = pd.read_csv("data/test_preprocessed.csv")

print("train_base shape:", train_base.shape)
print("val_base shape:", val_base.shape)
print("test_base shape:", test_base.shape)

train_base shape: (552070, 23)
val_base shape: (138018, 23)
test_base shape: (295753, 22)


In [3]:
feature_cols = [col for col in train_base.columns if col not in [ID_COL, TARGET_COL]]
missing_in_val = sorted(set(feature_cols) - set(val_base.columns))
missing_in_test = sorted(set(feature_cols) - set(test_base.columns))
extra_in_val = sorted(set(val_base.columns) - set(feature_cols) - {ID_COL, TARGET_COL})
extra_in_test = sorted(set(test_base.columns) - set(feature_cols) - {ID_COL})

print("feature columns:", len(feature_cols))
print("missing in val:", missing_in_val)
print("missing in test:", missing_in_test)
print("extra in val:", extra_in_val)
print("extra in test:", extra_in_test)

assert not missing_in_val
assert not missing_in_test
assert not extra_in_val
assert not extra_in_test

feature columns: 21
missing in val: []
missing in test: []
extra in val: []
extra in test: []


## Feature Engineering Helpers

In [4]:
def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    result = numerator / denominator.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan).fillna(0)


def add_bmi_category(df):
    bins = [-np.inf, 18.5, 25.0, 30.0, np.inf]
    labels = ["underweight", "normal", "overweight", "obese"]
    return pd.cut(df["bmi"], bins=bins, labels=labels, right=False).astype("object")


def add_engineered_features(df):
    df = df.copy()

    stress_map = {"low": 0, "medium": 1, "high": 2}
    sleep_quality_map = {"poor": 0, "average": 1, "good": 2}
    activity_map = {"sedentary": 0, "moderate": 1, "active": 2}
    smoking_alcohol_map = {"no": 0, "occasional": 1, "yes": 2}

    df["stress_score"] = df["stress_level"].map(stress_map).fillna(-1).astype(int)
    df["sleep_quality_score"] = df["sleep_quality"].map(sleep_quality_map).fillna(-1).astype(int)
    df["physical_activity_score"] = df["physical_activity_level"].map(activity_map).fillna(-1).astype(int)
    df["smoking_alcohol_score"] = df["smoking_alcohol"].map(smoking_alcohol_map).fillna(-1).astype(int)

    df["sleep_quality_risk"] = (2 - df["sleep_quality_score"]).clip(lower=0)
    df["activity_risk"] = (2 - df["physical_activity_score"]).clip(lower=0)

    df["sleep_deficit_from_8"] = (df["sleep_duration"] - 8).abs()
    df["sleep_outside_7_9"] = np.where(df["sleep_duration"] < 7, 7 - df["sleep_duration"], 0) + np.where(df["sleep_duration"] > 9, df["sleep_duration"] - 9, 0)
    df["is_low_sleep"] = (df["sleep_duration"] < 6).astype(int)
    df["is_high_sleep"] = (df["sleep_duration"] > 9).astype(int)

    df["bmi_category"] = add_bmi_category(df)
    df["bmi_risk_score"] = np.select(
        [df["bmi"] < 18.5, df["bmi"] < 25, df["bmi"] < 30, df["bmi"] >= 30],
        [1, 0, 1, 2],
        default=0,
    )
    df["is_high_bmi"] = (df["bmi"] >= 25).astype(int)
    df["is_obese_bmi"] = (df["bmi"] >= 30).astype(int)

    df["heart_rate_distance_from_75"] = (df["heart_rate"] - 75).abs()
    df["is_low_heart_rate"] = (df["heart_rate"] < 60).astype(int)
    df["is_high_heart_rate"] = (df["heart_rate"] > 90).astype(int)

    df["is_low_water"] = (df["water_intake"] < 1.5).astype(int)
    df["is_high_water"] = (df["water_intake"] > 3.0).astype(int)

    df["steps_per_exercise_min"] = safe_divide(df["step_count"], df["exercise_duration"] + 1)
    df["exercise_min_per_1000_steps"] = safe_divide(df["exercise_duration"], df["step_count"] / 1000 + 1)
    df["calories_per_step"] = safe_divide(df["calorie_expenditure"], df["step_count"] + 1)
    df["calories_per_exercise_min"] = safe_divide(df["calorie_expenditure"], df["exercise_duration"] + 1)
    df["water_per_1000_calories"] = safe_divide(df["water_intake"], df["calorie_expenditure"] / 1000)
    df["water_per_exercise_min"] = safe_divide(df["water_intake"], df["exercise_duration"] + 1)

    df["diet_is_balanced"] = (df["diet_type"] == "balanced").astype(int)
    df["diet_is_veg"] = (df["diet_type"] == "veg").astype(int)
    df["diet_is_non_veg"] = (df["diet_type"] == "non-veg").astype(int)

    df["poor_sleep_high_stress"] = ((df["sleep_quality_risk"] >= 2) & (df["stress_score"] >= 2)).astype(int)
    df["low_activity_high_bmi"] = ((df["activity_risk"] >= 2) & (df["bmi"] >= 25)).astype(int)
    df["low_sleep_high_stress"] = ((df["is_low_sleep"] == 1) & (df["stress_score"] >= 2)).astype(int)
    df["high_bmi_low_activity"] = ((df["is_high_bmi"] == 1) & (df["activity_risk"] >= 1)).astype(int)

    df["lifestyle_risk_score"] = (
        df["stress_score"].clip(lower=0)
        + df["sleep_quality_risk"]
        + df["activity_risk"]
        + df["smoking_alcohol_score"].clip(lower=0)
        + df["is_low_sleep"]
        + df["is_low_water"]
    )
    df["metabolic_risk_score"] = (
        df["bmi_risk_score"]
        + df["is_high_heart_rate"]
        + df["is_low_heart_rate"]
        + df["is_high_bmi"]
        + df["activity_risk"]
    )
    df["recovery_score"] = (
        df["sleep_quality_score"].clip(lower=0)
        + df["physical_activity_score"].clip(lower=0)
        + (1 - df["is_low_sleep"])
        + (1 - df["is_low_water"])
    )

    return df

## Apply Formula Features To Train, Validation, And Test

In [5]:
train_fe = add_engineered_features(train_base)
val_fe = add_engineered_features(val_base)
test_fe = add_engineered_features(test_base)

new_feature_cols = [col for col in train_fe.columns if col not in train_base.columns]

print("new feature count:", len(new_feature_cols))
print("train_fe shape:", train_fe.shape)
print("val_fe shape:", val_fe.shape)
print("test_fe shape:", test_fe.shape)
for col in new_feature_cols:
    print("-", col)

new feature count: 35
train_fe shape: (552070, 58)
val_fe shape: (138018, 58)
test_fe shape: (295753, 57)
- stress_score
- sleep_quality_score
- physical_activity_score
- smoking_alcohol_score
- sleep_quality_risk
- activity_risk
- sleep_deficit_from_8
- sleep_outside_7_9
- is_low_sleep
- is_high_sleep
- bmi_category
- bmi_risk_score
- is_high_bmi
- is_obese_bmi
- heart_rate_distance_from_75
- is_low_heart_rate
- is_high_heart_rate
- is_low_water
- is_high_water
- steps_per_exercise_min
- exercise_min_per_1000_steps
- calories_per_step
- calories_per_exercise_min
- water_per_1000_calories
- water_per_exercise_min
- diet_is_balanced
- diet_is_veg
- diet_is_non_veg
- poor_sleep_high_stress
- low_activity_high_bmi
- low_sleep_high_stress
- high_bmi_low_activity
- lifestyle_risk_score
- metabolic_risk_score
- recovery_score


In [6]:
train_only_features = sorted(set(train_fe.columns) - set(val_fe.columns) - {TARGET_COL})
val_only_features = sorted(set(val_fe.columns) - set(train_fe.columns))
test_only_features = sorted(set(test_fe.columns) - set(train_fe.columns))
missing_in_test_after_fe = sorted(set(train_fe.columns) - set(test_fe.columns) - {TARGET_COL})

print("train-only feature columns excluding target compared to val:", train_only_features)
print("val-only feature columns:", val_only_features)
print("test-only feature columns:", test_only_features)
print("missing in test excluding target:", missing_in_test_after_fe)

assert not train_only_features
assert not val_only_features
assert not test_only_features
assert not missing_in_test_after_fe
assert TARGET_COL not in test_fe.columns

train-only feature columns excluding target compared to val: []
val-only feature columns: []
test-only feature columns: []
missing in test excluding target: []


## Target-Based Feature Review On Train Only

These summaries use `health_condition` only to evaluate candidate features. Target values are not used to create feature values.

In [7]:
engineered_numeric_cols = [
    col for col in new_feature_cols
    if pd.api.types.is_numeric_dtype(train_fe[col])
]

feature_target_summary = train_fe.groupby(TARGET_COL)[engineered_numeric_cols].agg(["mean", "std"]).T
feature_target_summary

health_condition                      at-risk         fit   unhealthy
stress_score                mean     0.989872    0.170812    1.849997
                            std      0.663759    0.415383    0.378568
sleep_quality_score         mean     1.011263    1.239621    0.607939
                            std      0.777991    0.741644    0.673665
physical_activity_score     mean     0.922126    1.939577    1.005067
                            std      0.796168    0.292368    0.808413
smoking_alcohol_score       mean     1.040876    0.815652    1.269040
                            std      0.826341    0.805782    0.782809
sleep_quality_risk          mean     0.988737    0.760379    1.392061
                            std      0.777991    0.741644    0.673665
activity_risk               mean     1.077874    0.060423    0.994933
                            std      0.796168    0.292368    0.808413
sleep_deficit_from_8        mean     1.180811    0.662014    2.463804
                            std      0.788990    0.457642    0.787992
sleep_outside_7_9           mean     0.386378    0.063206    1.469537
                            std      0.607138    0.228139    0.774880
is_low_sleep                mean     0.142945    0.007506    0.858659
                            std      0.350017    0.086312    0.348377
is_high_sleep               mean     0.050041    0.093650    0.001819
                            std      0.218030    0.291346    0.042611
bmi_risk_score              mean     0.233178    0.206174    0.345525
                            std      0.424006    0.404563    0.526009
is_high_bmi                 mean     0.200743    0.084982    0.320254
                            std      0.400557    0.278859    0.466579
is_obese_bmi                mean     0.000487    0.000000    0.025271
                            std      0.022069    0.000000    0.156949
heart_rate_distance_from_75 mean     6.528613    6.543602    6.340848
                            std      4.881610    4.880859    4.762040
is_low_heart_rate           mean     0.030039    0.025909    0.023517
                            std      0.170695    0.158867    0.151541
is_high_heart_rate          mean     0.034372    0.038785    0.035254
                            std      0.182183    0.193086    0.184423
is_low_water                mean     0.064339    0.067804    0.064510
                            std      0.245357    0.251412    0.245662
is_high_water               mean     0.070948    0.079518    0.070725
                            std      0.256739    0.270549    0.256367
steps_per_exercise_min      mean   381.444538  242.797667  371.887139
                            std   1001.290919  184.393607  974.307055
exercise_min_per_1000_steps mean     4.763177    4.165685    4.718338
                            std      2.962029    1.406203    2.865993
calories_per_step           mean     0.375283    0.217163    0.364314
                            std      0.319297    0.090561    0.310478
calories_per_exercise_min   mean   121.738628   50.442176  116.963519
                            std    331.736632   68.771654  321.996771
water_per_1000_calories     mean     1.011916    0.941924    0.998085
                            std      0.290244    0.256561    0.284450
water_per_exercise_min      mean     0.127542    0.047791    0.121389
                            std      0.367345    0.086175    0.355284
diet_is_balanced            mean     0.326426    0.349256    0.341519
                            std      0.468906    0.476742    0.474224
diet_is_veg                 mean     0.344154    0.358771    0.343663
                            std      0.475092    0.479647    0.474935
diet_is_non_veg             mean     0.329420    0.291973    0.314818
                            std      0.470003    0.454677    0.464449
poor_sleep_high_stress      mean     0.054802    0.004931    0.432578
                            std      0.227594    0.070046    0.495439
low_activi

In [8]:
target_counts_by_bmi_category = pd.crosstab(
    train_fe["bmi_category"],
    train_fe[TARGET_COL],
    normalize="index",
).round(4)

target_counts_by_bmi_category

health_condition,at-risk,fit,unhealthy
bmi_category,,,
normal,0.8652,0.0601,0.0747
obese,0.1652,0.0000,0.8348
overweight,0.8532,0.0243,0.1224
underweight,0.7969,0.2031,0.0000


In [9]:
binary_like_cols = [
    col for col in engineered_numeric_cols
    if set(train_fe[col].dropna().unique()).issubset({0, 1})
]

binary_feature_rates_by_target = train_fe.groupby(TARGET_COL)[binary_like_cols].mean().T.sort_index()
binary_feature_rates_by_target.style.format("{:.3f}")

health_condition,at-risk,fit,unhealthy
diet_is_balanced,0.326,0.349,0.342
diet_is_non_veg,0.329,0.292,0.315
diet_is_veg,0.344,0.359,0.344
high_bmi_low_activity,0.145,0.004,0.215
is_high_bmi,0.201,0.085,0.320
is_high_heart_rate,0.034,0.039,0.035
is_high_sleep,0.050,0.094,0.002
is_high_water,0.071,0.080,0.071
is_low_heart_rate,0.030,0.026,0.024
is_low_sleep,0.143,0.008,0.859


## Numeric Encoding For Modeling

One-hot category levels are learned from train split only. Validation and test receive the same encoded columns. Unknown validation/test categories become all zeros for that categorical field.

In [10]:
def safe_category_name(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^0-9a-zA-Z]+", "_", value).strip("_")
    return value or "missing"


categorical_cols_to_encode = [
    col for col in train_fe.columns
    if col != TARGET_COL and not pd.api.types.is_numeric_dtype(train_fe[col])
]

category_levels = {
    col: sorted(train_fe[col].dropna().astype(str).unique().tolist())
    for col in categorical_cols_to_encode
}


def add_one_hot_from_train_levels(df, category_levels):
    df = df.copy()
    encoded_cols = []

    for col, levels in category_levels.items():
        values = df[col].astype(str)
        for level in levels:
            encoded_col = f"{col}__{safe_category_name(level)}"
            df[encoded_col] = (values == level).astype(int)
            encoded_cols.append(encoded_col)

    return df, encoded_cols


train_fe_encoded, encoded_categorical_cols = add_one_hot_from_train_levels(train_fe, category_levels)
val_fe_encoded, _ = add_one_hot_from_train_levels(val_fe, category_levels)
test_fe_encoded, _ = add_one_hot_from_train_levels(test_fe, category_levels)

train_features_numeric = train_fe_encoded.drop(columns=categorical_cols_to_encode)
val_features_numeric = val_fe_encoded.drop(columns=[col for col in categorical_cols_to_encode if col in val_fe_encoded.columns])
test_features_numeric = test_fe_encoded.drop(columns=[col for col in categorical_cols_to_encode if col in test_fe_encoded.columns])

train_non_numeric = train_features_numeric.drop(columns=[TARGET_COL]).select_dtypes(exclude="number").columns.tolist()
val_non_numeric = val_features_numeric.drop(columns=[TARGET_COL]).select_dtypes(exclude="number").columns.tolist()
test_non_numeric = test_features_numeric.select_dtypes(exclude="number").columns.tolist()

train_only_numeric_cols = sorted(set(train_features_numeric.columns) - set(val_features_numeric.columns) - {TARGET_COL})
val_only_numeric_cols = sorted(set(val_features_numeric.columns) - set(train_features_numeric.columns))
test_only_numeric_cols = sorted(set(test_features_numeric.columns) - set(train_features_numeric.columns))
missing_test_numeric_cols = sorted(set(train_features_numeric.columns) - set(test_features_numeric.columns) - {TARGET_COL})

print("categorical columns encoded:", categorical_cols_to_encode)
print("encoded categorical feature count:", len(encoded_categorical_cols))
print("non-numeric train feature columns excluding target:", train_non_numeric)
print("non-numeric val feature columns excluding target:", val_non_numeric)
print("non-numeric test feature columns:", test_non_numeric)
print("train-only numeric columns excluding target compared to val:", train_only_numeric_cols)
print("val-only numeric columns:", val_only_numeric_cols)
print("test-only numeric columns:", test_only_numeric_cols)
print("missing test numeric columns excluding target:", missing_test_numeric_cols)
print("train_features_numeric shape:", train_features_numeric.shape)
print("val_features_numeric shape:", val_features_numeric.shape)
print("test_features_numeric shape:", test_features_numeric.shape)

assert not train_non_numeric
assert not val_non_numeric
assert not test_non_numeric
assert not train_only_numeric_cols
assert not val_only_numeric_cols
assert not test_only_numeric_cols
assert not missing_test_numeric_cols
assert TARGET_COL not in test_features_numeric.columns

categorical columns encoded: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category']
encoded categorical feature count: 22
non-numeric train feature columns excluding target: []
non-numeric val feature columns excluding target: []
non-numeric test feature columns: []
train-only numeric columns excluding target compared to val: []
val-only numeric columns: []
test-only numeric columns: []
missing test numeric columns excluding target: []
train_features_numeric shape: (552070, 73)
val_features_numeric shape: (138018, 73)
test_features_numeric shape: (295753, 72)


## Mutual Information Check On Train Only

This is a train-only relevance ranking for later feature-selection experiments. It does not create target-derived features.

In [11]:
MI_SAMPLE_SIZE = 200_000
RANDOM_STATE = 42

model_feature_cols = [col for col in train_features_numeric.columns if col not in [ID_COL, TARGET_COL]]
mi_data = train_features_numeric[[TARGET_COL] + model_feature_cols].copy()
if len(mi_data) > MI_SAMPLE_SIZE:
    mi_data = mi_data.sample(MI_SAMPLE_SIZE, random_state=RANDOM_STATE)

X_mi = mi_data[model_feature_cols].copy()
y_mi = LabelEncoder().fit_transform(mi_data[TARGET_COL])

discrete_features = [X_mi[col].nunique() <= 20 for col in model_feature_cols]
X_mi = X_mi.fillna(X_mi.median(numeric_only=True))

mi_scores = mutual_info_classif(
    X_mi,
    y_mi,
    discrete_features=discrete_features,
    random_state=RANDOM_STATE,
)

mi_report = (
    pd.DataFrame({"feature": model_feature_cols, "mutual_information": mi_scores})
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

mi_report.head(50).style.format({"mutual_information": "{:.5f}"})

,feature,mutual_information
0,low_sleep_high_stress,0.18160
1,sleep_duration,0.15480
2,sleep_deficit_from_8,0.14421
3,sleep_outside_7_9,0.14066
4,lifestyle_risk_score,0.13966
5,stress_score,0.12350
6,is_low_sleep,0.10423
7,stress_level__high,0.08385
8,recovery_score,0.06859
9,stress_level__low,0.06735


## Save Feature Datasets

In [12]:
train_features_path = "data/train_split_features.csv"
val_features_path = "data/val_split_features.csv"
test_features_path = "data/test_features.csv"
train_numeric_path = "data/train_split_features_numeric.csv"
val_numeric_path = "data/val_split_features_numeric.csv"
test_numeric_path = "data/test_features_numeric.csv"

train_fe.to_csv(train_features_path, index=False)
val_fe.to_csv(val_features_path, index=False)
test_fe.to_csv(test_features_path, index=False)
train_features_numeric.to_csv(train_numeric_path, index=False)
val_features_numeric.to_csv(val_numeric_path, index=False)
test_features_numeric.to_csv(test_numeric_path, index=False)

print("saved:", train_features_path, train_fe.shape)
print("saved:", val_features_path, val_fe.shape)
print("saved:", test_features_path, test_fe.shape)
print("saved:", train_numeric_path, train_features_numeric.shape)
print("saved:", val_numeric_path, val_features_numeric.shape)
print("saved:", test_numeric_path, test_features_numeric.shape)

saved: data/train_split_features.csv (552070, 58)
saved: data/val_split_features.csv (138018, 58)
saved: data/test_features.csv (295753, 57)
saved: data/train_split_features_numeric.csv (552070, 73)
saved: data/val_split_features_numeric.csv (138018, 73)
saved: data/test_features_numeric.csv (295753, 72)


In [13]:
train_features_numeric.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier,stress_score,sleep_quality_score,physical_activity_score,smoking_alcohol_score,sleep_quality_risk,activity_risk,sleep_deficit_from_8,sleep_outside_7_9,is_low_sleep,is_high_sleep,bmi_risk_score,is_high_bmi,is_obese_bmi,heart_rate_distance_from_75,is_low_heart_rate,is_high_heart_rate,is_low_water,is_high_water,steps_per_exercise_min,exercise_min_per_1000_steps,calories_per_step,calories_per_exercise_min,water_per_1000_calories,water_per_exercise_min,diet_is_balanced,diet_is_veg,diet_is_non_veg,poor_sleep_high_stress,low_activity_high_bmi,low_sleep_high_stress,high_bmi_low_activity,lifestyle_risk_score,metabolic_risk_score,recovery_score,diet_type__balanced,diet_type__non_veg,diet_type__veg,stress_level__high,stress_level__low,stress_level__medium,sleep_quality__average,sleep_quality__good,sleep_quality__poor,physical_activity_level__active,physical_activity_level__moderate,physical_activity_level__sedentary,smoking_alcohol__no,smoking_alcohol__occasional,smoking_alcohol__yes,gender__female,gender__male,gender__other,bmi_category__normal,bmi_category__obese,bmi_category__overweight,bmi_category__underweight
0,313415,at-risk,7.17,91.3,26.86,2635.0,1398.0,49.1,2.03,0,0,0,0,0,0,0,0,1,1,0,2,1,2,0.83,0.00,0,0,1,1,0,16.3,0,1,0,0,27.904192,20.475396,1.883488,52.594810,0.770398,0.040519,0,0,1,0,1,0,1,6,5,3,0,1,0,0,0,1,1,0,0,0,0,1,0,0,1,0,1,0,0,0,1,0
1,3515,at-risk,6.99,75.1,24.23,2382.0,13466.0,50.8,1.83,0,0,0,0,0,0,0,0,0,1,2,1,1,0,1.01,0.01,0,0,0,0,0,0.1,0,0,0,0,259.961390,3.511683,0.176877,45.984556,0.768262,0.035328,1,0,0,0,0,0,0,2,0,5,1,0,0,0,1,0,1,0,0,1,0,0,0,1,0,0,1,0,1,0,0,0
2,501194,at-risk,8.66,82.4,21.41,2314.0,9473.0,23.3,3.09,0,0,0,0,0,0,0,0,0,0,0,2,2,2,0.66,0.00,0,0,0,0,0,7.4,0,0,0,1,389.835391,2.224768,0.244247,95.226337,1.335350,0.127160,1,0,0,0,0,0,0,6,2,2,1,0,0,0,1,0,0,0,1,0,0,1,0,0,1,1,0,0,1,0,0,0
3,303602,at-risk,6.99,74.5,22.59,2165.0,7052.0,21.6,1.92,0,0,0,0,0,0,0,0,1,1,0,1,1,2,1.01,0.01,0,0,0,0,0,0.5,0,0,0,0,312.035398,2.682563,0.306962,95.796460,0.886836,0.084956,0,1,0,0,0,0,0,5,2,3,0,0,1,0,0,1,1,0,0,0,0,1,0,1,0,0,1,0,1,0,0,0
4,117943,at-risk,8.83,68.2,22.01,2108.0,13521.0,52.9,2.35,0,0,0,0,0,0,0,0,1,1,2,2,1,0,0.83,0.00,0,0,0,0,0,6.8,0,0,0,0,250.853432,3.643000,0.155894,39.109462,1.114801,0.043599,0,0,1,0,0,0,0,4,0,5,0,1,0,0,0,1,1,0,0,1,0,0,0,0,1,0,1,0,1,0,0,0


In [14]:
val_features_numeric.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier,stress_score,sleep_quality_score,physical_activity_score,smoking_alcohol_score,sleep_quality_risk,activity_risk,sleep_deficit_from_8,sleep_outside_7_9,is_low_sleep,is_high_sleep,bmi_risk_score,is_high_bmi,is_obese_bmi,heart_rate_distance_from_75,is_low_heart_rate,is_high_heart_rate,is_low_water,is_high_water,steps_per_exercise_min,exercise_min_per_1000_steps,calories_per_step,calories_per_exercise_min,water_per_1000_calories,water_per_exercise_min,diet_is_balanced,diet_is_veg,diet_is_non_veg,poor_sleep_high_stress,low_activity_high_bmi,low_sleep_high_stress,high_bmi_low_activity,lifestyle_risk_score,metabolic_risk_score,recovery_score,diet_type__balanced,diet_type__non_veg,diet_type__veg,stress_level__high,stress_level__low,stress_level__medium,sleep_quality__average,sleep_quality__good,sleep_quality__poor,physical_activity_level__active,physical_activity_level__moderate,physical_activity_level__sedentary,smoking_alcohol__no,smoking_alcohol__occasional,smoking_alcohol__yes,gender__female,gender__male,gender__other,bmi_category__normal,bmi_category__obese,bmi_category__overweight,bmi_category__underweight
0,304516,at-risk,6.82,67.9,27.17,2464.0,13456.0,39.9,2.05,0,0,0,0,0,0,0,0,1,2,1,0,0,1,1.18,0.18,0,0,1,1,0,7.1,0,0,0,0,328.997555,2.760100,0.183102,60.244499,0.831981,0.050122,0,1,0,0,0,0,1,2,3,5,0,0,1,0,0,1,0,1,0,0,1,0,1,0,0,0,0,1,0,0,1,0
1,165358,at-risk,7.97,92.9,16.86,2240.0,7456.0,26.9,2.14,0,0,0,0,0,0,0,0,2,2,1,0,0,1,0.03,0.00,0,0,1,0,0,17.9,0,1,0,0,267.240143,3.181173,0.300389,80.286738,0.955357,0.076703,1,0,0,0,0,0,0,3,3,5,1,0,0,1,0,0,0,1,0,0,1,0,1,0,0,1,0,0,0,0,0,1
2,671841,at-risk,6.99,75.1,22.18,2352.0,4140.0,19.7,2.33,0,0,0,0,0,0,0,0,0,0,0,1,2,2,1.01,0.01,0,0,0,0,0,0.1,0,0,0,0,200.000000,3.832685,0.567979,113.623188,0.990646,0.112560,0,0,1,0,0,0,0,5,2,2,0,1,0,0,1,0,0,0,1,0,0,1,0,1,0,0,1,0,1,0,0,0
3,403460,at-risk,6.99,83.1,21.93,2620.0,11656.0,41.1,1.22,0,0,0,0,0,0,0,0,1,2,1,1,0,1,1.01,0.01,0,0,0,0,0,8.1,0,0,1,0,276.864608,3.247472,0.224758,62.232779,0.465649,0.028979,0,0,1,0,0,0,0,4,1,4,0,1,0,0,0,1,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0
4,351133,at-risk,8.14,80.0,23.41,2442.0,4484.0,38.9,2.32,0,0,0,0,0,0,0,0,1,1,0,0,1,2,0.14,0.00,0,0,0,0,0,5.0,0,0,0,0,112.380952,7.093363,0.544482,61.203008,0.950041,0.058145,0,1,0,0,0,0,0,4,2,3,0,0,1,0,0,1,1,0,0,0,0,1,1,0,0,0,0,1,1,0,0,0


In [15]:
test_features_numeric.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier,stress_score,sleep_quality_score,physical_activity_score,smoking_alcohol_score,sleep_quality_risk,activity_risk,sleep_deficit_from_8,sleep_outside_7_9,is_low_sleep,is_high_sleep,bmi_risk_score,is_high_bmi,is_obese_bmi,heart_rate_distance_from_75,is_low_heart_rate,is_high_heart_rate,is_low_water,is_high_water,steps_per_exercise_min,exercise_min_per_1000_steps,calories_per_step,calories_per_exercise_min,water_per_1000_calories,water_per_exercise_min,diet_is_balanced,diet_is_veg,diet_is_non_veg,poor_sleep_high_stress,low_activity_high_bmi,low_sleep_high_stress,high_bmi_low_activity,lifestyle_risk_score,metabolic_risk_score,recovery_score,diet_type__balanced,diet_type__non_veg,diet_type__veg,stress_level__high,stress_level__low,stress_level__medium,sleep_quality__average,sleep_quality__good,sleep_quality__poor,physical_activity_level__active,physical_activity_level__moderate,physical_activity_level__sedentary,smoking_alcohol__no,smoking_alcohol__occasional,smoking_alcohol__yes,gender__female,gender__male,gender__other,bmi_category__normal,bmi_category__obese,bmi_category__overweight,bmi_category__underweight
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,0,0,0,0,0,0,0,0,2,0,2,1,2,0,2.65,1.65,1,0,0,0,0,10.1,0,0,0,0,234.165289,3.922991,0.193746,45.371901,0.677596,0.030744,0,1,0,1,0,1,0,6,0,3,0,0,1,1,0,0,0,0,1,1,0,0,0,1,0,0,1,0,1,0,0,0
1,690089,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,0,0,0,0,0,0,0,0,2,0,0,2,2,2,1.01,0.01,0,0,0,0,0,8.1,0,0,0,0,266.705882,3.140623,0.260659,69.529412,1.353638,0.094118,1,0,0,1,0,0,0,8,2,2,1,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,0,0
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,0,0,0,1,0,0,0,1,1,0,2,0,2,0,1.32,0.32,0,0,0,0,0,15.3,1,0,0,0,267.676768,3.403509,0.229417,61.414141,0.907895,0.055758,1,0,0,0,0,0,0,3,1,4,1,0,0,0,0,1,0,0,1,1,0,0,1,0,0,0,1,0,1,0,0,0
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,0,0,0,0,0,0,0,0,0,2,1,2,0,1,0.87,0.00,0,0,1,1,0,3.5,0,0,0,0,109.343696,7.761560,0.393872,43.074266,0.938252,0.040415,0,1,0,0,0,0,1,3,3,5,0,0,1,0,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0,0,1,0
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,0,0,0,0,0,0,0,0,2,1,2,1,1,0,2.51,1.51,1,0,0,0,0,2.7,0,0,0,0,343.910891,2.645361,0.131558,45.247525,1.340263,0.060644,0,1,0,0,0,1,0,5,0,4,0,0,1,1,0,0,1,0,0,1,0,0,0,1,0,0,0,1,1,0,0,0
